In [ ]:
# 2D Wave Equation in 3D

This notebook solves the 2D wave equation with a finite-difference scheme and animates the result as a 3D surface with `N_molecules` sampled points moving with the wave field.

In [ ]:
import matplotlib
matplotlib.use('Agg')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# -----------------------------
# Parameters
# -----------------------------
Lx = Ly = 10.0
nx = ny = 61
dx = Lx / (nx - 1)
dy = Ly / (ny - 1)

c = 1.0                 # wave speed
dt = 0.35 * min(dx, dy) / c
nt = 120                # number of time steps
N_molecules = 40        # sampled particles shown on top of the field

x = np.linspace(0.0, Lx, nx)
y = np.linspace(0.0, Ly, ny)
X, Y = np.meshgrid(x, y)

# -----------------------------
# Initial conditions: a smooth Gaussian pulse
# -----------------------------
u_prev = np.exp(-((X - 5.0) ** 2 + (Y - 5.0) ** 2) / 1.25)
u_curr = u_prev.copy()

# Molecule sample points on the grid
rng = np.random.default_rng(7)
particle_idx = rng.choice(nx * ny, size=N_molecules, replace=False)
particle_iy, particle_ix = np.unravel_index(particle_idx, (ny, nx))
particle_x = X[particle_iy, particle_ix]
particle_y = Y[particle_iy, particle_ix]

# Discrete Laplacian helper
coef_x = (c * dt / dx) ** 2
coef_y = (c * dt / dy) ** 2


def step_wave(u_now, u_before):
    u_next = np.zeros_like(u_now)
    center = u_now[1:-1, 1:-1]
    lap_x = u_now[1:-1, 2:] - 2.0 * center + u_now[1:-1, :-2]
    lap_y = u_now[2:, 1:-1] - 2.0 * center + u_now[:-2, 1:-1]
    u_next[1:-1, 1:-1] = (
        2.0 * center
        - u_before[1:-1, 1:-1]
        + coef_x * lap_x
        + coef_y * lap_y
    )
    return u_next


# -----------------------------
# Build frames
# -----------------------------
frames = [u_curr.copy()]
for _ in range(nt):
    u_next = step_wave(u_curr, u_prev)
    u_prev, u_curr = u_curr, u_next
    frames.append(u_curr.copy())

# -----------------------------
# 3D animation
# -----------------------------
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
ax.set_xlim(0, Lx)
ax.set_ylim(0, Ly)
ax.set_zlim(-1.2, 1.2)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('u(x, y, t)')
ax.set_title('2D Wave Equation with N Molecule Points')

surface = [ax.plot_surface(X, Y, frames[0], cmap='viridis', edgecolor='none', alpha=0.85)]
particles = ax.scatter(
    particle_x,
    particle_y,
    frames[0][particle_iy, particle_ix],
    s=28,
    c='crimson',
    depthshade=True,
)


def update(frame_index):
    surface[0].remove()
    surface[0] = ax.plot_surface(
        X,
        Y,
        frames[frame_index],
        cmap='viridis',
        edgecolor='none',
        alpha=0.85,
    )
    particles._offsets3d = (
        particle_x,
        particle_y,
        frames[frame_index][particle_iy, particle_ix],
    )
    ax.set_title(f'2D Wave Equation with N Molecule Points | frame={frame_index}')
    return surface[0], particles

anim = FuncAnimation(fig, update, frames=len(frames), interval=35, blit=False)
HTML(anim.to_jshtml())

Animation size has reached 21037863 bytes, exceeding the limit of 20971520.0. If you're sure you want a larger animation embedded, set the animation.embed_limit rc parameter to a larger value (in MB). This and further frames will be dropped.


In [ ]:
import numpy as np

def create_grid(
        sr: int = 44100, duration: float = 0.020, # 20 ms
        gamma: float = 24000,
        # dgamma: int = 10, gamma_lim : tuple = (10000, 40000),
        dalpha: int = 100, alpha_lim : tuple = (-0.05, 0.2),
        dbeta: int = 100, beta_lim : tuple = (-0.2, 1.5),
    ):
    # Create 1D coordinate arrays
    # gamma_array = np.linspace(gamma_lim[0], gamma_lim[1], dgamma)
    alpha_array = np.linspace(alpha_lim[0], alpha_lim[1], dalpha)
    beta_array = np.linspace(beta_lim[0], beta_lim[1], dbeta)

    # Generate the 2D coordinate mesh matrices
    # ALPHA, BETA = np.meshgrid(alpha, beta)

    # Create the time axis
    time_array = np.linspace(0, duration, int(sr * duration))
    # alpha_array = np.ones_like(time_array) * alpha

    for alpha in alpha_array:
        for beta in beta_array:
            alpha_cte = alpha * np.ones(int(sr * duration))
            beta_cte = beta * np.ones(int(sr * duration))

            # MODELo
    # return ALPHA, BETA


alpha, beta = create_grid()
alpha

array([[-0.05      , -0.04747475, -0.04494949, ...,  0.19494949,
         0.19747475,  0.2       ],
       [-0.05      , -0.04747475, -0.04494949, ...,  0.19494949,
         0.19747475,  0.2       ],
       [-0.05      , -0.04747475, -0.04494949, ...,  0.19494949,
         0.19747475,  0.2       ],
       ...,
       [-0.05      , -0.04747475, -0.04494949, ...,  0.19494949,
         0.19747475,  0.2       ],
       [-0.05      , -0.04747475, -0.04494949, ...,  0.19494949,
         0.19747475,  0.2       ],
       [-0.05      , -0.04747475, -0.04494949, ...,  0.19494949,
         0.19747475,  0.2       ]], shape=(100, 100))